# Deep Research Agent — Evaluation Analysis

This notebook loads eval results from `eval/results/` and produces:
1. The main ablation comparison table
2. Per-category breakdown
3. Failure mode analysis
4. Unanswerable question behavior

Run the harness first: `python -m eval.harness`

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import defaultdict
import statistics

import pandas as pd

# Allow imports from repo root
sys.path.insert(0, str(Path.cwd().parent))

RESULTS_DIR = Path("../eval/results")

In [ ]:
def load_latest_results() -> list[dict]:
    """Load the most recent eval run."""
    result_files = sorted(RESULTS_DIR.glob("*.json"))
    if not result_files:
        raise FileNotFoundError(
            "No eval results found. Run: python -m eval.harness"
        )
    latest = result_files[-1]
    print(f"Loading: {latest.name}")
    with open(latest) as f:
        data = json.load(f)
    return data["results"]

results = load_latest_results()
df = pd.DataFrame([
    {
        "task_id": r["task_id"],
        "category": r["category"],
        "config": r["config"],
        "citation_accuracy": r["scores"]["citation_accuracy"] if r.get("scores") else None,
        "completeness": r["scores"]["completeness"] if r.get("scores") else None,
        "hallucination_rate": r["scores"]["hallucination_rate"] if r.get("scores") else None,
        "tool_calls": r.get("tool_calls"),
        "uncertainty_reported": r["scores"].get("uncertainty_reported") if r.get("scores") else None,
        "error": r.get("error"),
    }
    for r in results
])

print(f"Total runs: {len(df)}")
print(f"Tasks: {df['task_id'].nunique()} | Configs: {df['config'].nunique()}")
df.head()

## 1. Main Ablation Table

This is the key result: how much does each component (planning, verification) contribute?

In [ ]:
config_order = ["no_plan_no_verify", "plan_no_verify", "plan_verify"]
config_labels = {
    "no_plan_no_verify": "No planning, no verification",
    "plan_no_verify": "Planning, no verification",
    "plan_verify": "Planning + verification",
}

agg = (
    df[df["error"].isna()]
    .groupby("config")[["citation_accuracy", "completeness", "hallucination_rate", "tool_calls"]]
    .mean()
    .reindex(config_order)
)

# Compute relative improvements vs baseline
baseline = agg.loc["no_plan_no_verify"]

display_df = pd.DataFrame({
    "Configuration": [config_labels[c] for c in config_order],
    "Citation Acc.": [f"{agg.loc[c, 'citation_accuracy']:.2f}" for c in config_order],
    "Completeness": [f"{agg.loc[c, 'completeness']:.2f}" for c in config_order],
    "Hallucination %": [f"{agg.loc[c, 'hallucination_rate']*100:.1f}%" for c in config_order],
    "Avg Tool Calls": [f"{agg.loc[c, 'tool_calls']:.1f}" for c in config_order],
})

display_df.set_index("Configuration", inplace=True)
print("\n=== ABLATION RESULTS ===")
display_df

## 2. Per-Category Breakdown

In [ ]:
# Full pipeline only, broken down by category
full_pipeline = df[(df["config"] == "plan_verify") & df["error"].isna()]

cat_agg = (
    full_pipeline
    .groupby("category")[["citation_accuracy", "completeness", "hallucination_rate", "tool_calls"]]
    .mean()
)

print("\n=== RESULTS BY CATEGORY (full pipeline) ===")
cat_agg.round(3)

## 3. Unanswerable Question Analysis

Key test: does the agent hallucinate on questions with no public answer, or does it correctly report uncertainty?

In [ ]:
unanswerable = df[
    (df["category"] == "unanswerable") & 
    (df["config"] == "plan_verify") & 
    df["error"].isna()
]

if len(unanswerable) > 0:
    uncertainty_rate = unanswerable["uncertainty_reported"].mean()
    hallucination_rate = unanswerable["hallucination_rate"].mean()
    
    print(f"Unanswerable tasks: {len(unanswerable)}")
    print(f"Uncertainty correctly reported: {uncertainty_rate:.0%}")
    print(f"Mean hallucination rate (should be low): {hallucination_rate:.1%}")
    print()
    
    unanswerable[["task_id", "hallucination_rate", "uncertainty_reported", "completeness"]]
else:
    print("No unanswerable task results found.")

## 4. Failure Mode Analysis

Examining the cases where the agent struggled most.

In [ ]:
# Tasks with high hallucination rate in full pipeline
full_pipeline_detailed = pd.DataFrame([
    {
        "task_id": r["task_id"],
        "category": r["category"],
        "question": r["question"][:80] + "...",
        "hallucination_rate": r["scores"]["hallucination_rate"] if r.get("scores") else None,
        "completeness": r["scores"]["completeness"] if r.get("scores") else None,
        "num_sources": r.get("num_sources", 0),
        "unanswered_sqs": len(r.get("unanswered_sub_questions", [])),
    }
    for r in results
    if r["config"] == "plan_verify" and not r.get("error")
])

if len(full_pipeline_detailed) > 0:
    print("Top 5 hardest tasks (highest hallucination rate):")
    print(
        full_pipeline_detailed
        .sort_values("hallucination_rate", ascending=False)
        .head(5)[["task_id", "category", "hallucination_rate", "completeness", "num_sources"]]
        .to_string(index=False)
    )
    print()
    print("Lowest completeness tasks:")
    print(
        full_pipeline_detailed
        .sort_values("completeness")
        .head(5)[["task_id", "category", "completeness", "unanswered_sqs", "num_sources"]]
        .to_string(index=False)
    )

## 5. Context Efficiency

Semantic compression ratio: how much of retrieved content actually ends up in the final prompt?

In [ ]:
# Load full results for compression analysis
def load_full_results() -> list[dict]:
    result_files = sorted(RESULTS_DIR.glob("*.json"))
    if not result_files:
        return []
    with open(result_files[-1]) as f:
        return json.load(f)["results"]

full_results = load_full_results()

tool_call_stats = defaultdict(list)
for r in full_results:
    if not r.get("error") and r.get("tool_calls"):
        tool_call_stats[r["config"]].append(r["tool_calls"])

print("API call breakdown by config:")
for config, calls in tool_call_stats.items():
    print(f"  {config}: mean={statistics.mean(calls):.1f}, "
          f"min={min(calls)}, max={max(calls)}")
print()
print("Note: Each tool call includes:")
print("  - 1 planner call")
print("  - Per sub-question: 1 Tavily search + N×Claude extraction calls")
print("  - 1 synthesis call")
print("  - 2 verification calls (extract_claims + verify)")